# Can you find Wally? 🔍

In the last workshop we looked at how a convolutional neural network (CNN) can tell us what is in an image. Today we go one step further and ask where an object is.

We will build a CNN from scratch in PyTorch that looks at an image and draws a box around Wally.

<br>

**To-dos:**

- **Making our own training and testing data**: we will randomly paste Wally onto hundreds of different backgrounds. Because we placed him ourselves, we know where he truly is in each image (the true bounding box).
- **Building a CNN from scratch**: convolution, pooling and outputs four numbers (the corners of Wally's box).
- **Writing the training loop ourselves**: forward pass, loss, `zero_grad`, `backward`, `step`.
- **Measuring how good the model's bounding boxes are**: Using Intersection over Union (IoU).
- **Testing on images the model has never seen**: green box = where Wally really is, red box = where the model thinks he is.

<hr>

# The structure of this notebook 📖

Work through the sections in order. Each code cell does one small job, and there are explanations above each cell!

Replace each `...` marked by a `# TODO` comment with your code before running that cell. Some blanks need a value, some need an expression, and some need a whole line.

<br>

**Tech stack used**

| Job | Library | Purpose |
|---|---|---|
| Model, loss, training loop | [PyTorch](https://pytorch.org/) (`torch`, `torch.nn`) | We write the forward pass, `zero_grad`, `backward` and `step` ourselves |
| Batching and splitting | `torch.utils.data` | `Dataset`, `DataLoader`, `random_split` |
| Making the synthetic images | [Pillow](https://pillow.readthedocs.io/) and [NumPy](https://numpy.org/) | Pasting Wally using his transparency (alpha) mask so that only Wally's pixels replace the background. Otherwise, the whole rectangular image of Wally will be pasted. |
| Plotting boxes and curves | [Matplotlib](https://matplotlib.org/) | `patches.Rectangle` for the true and predicted boxes |
| IoU metric | [torchvision](https://pytorch.org/vision/) | `torchvision.ops.box_iou`, which takes boxes as `(x1, y1, x2, y2)` |

<br>

**Want your model to train faster? ⚡:** Go to **Settings → Accelerator** and pick a **GPU**.

<hr>

# Step 0: Setting up ⚙️

## Task 0.0: Import the notebook and add the dataset 📂

Before running the code, we need to open this notebook in Kaggle and add the dataset containing our backgrounds and Wally cut-outs.

**Your Job**

1. Go to [Kaggle.com](https://www.kaggle.com/) and sign in (or create an account).
2. Create a **New Notebook**.
3. Download this workshop notebook (`.ipynb`) from **WhatsApp**.
4. In your new Kaggle notebook, go to **File → Import Notebook** and open your downloaded notebook.
5. In the right sidebar, go to **Input → Add Input** and paste the dataset link below into the search box.
6. Select the **Waldo Dataset** by **zekikam** and add it to your notebook.

The dataset can be found here:

https://www.kaggle.com/datasets/zekikam/waldo-dataset

Check that the dataset appears under **Input** before continuing. It contains the background images and transparent Wally pictures used in the next steps. Adding it makes these files available to the notebook under `/kaggle/input`.


In [ ]:
import os 
import csv
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageEnhance

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.ops import box_iou

### 0.1 The directory of our file and the variables we might want to change

Putting all the settings in one place means you never have to hunt through the notebook to change something.

- The **backgrounds** are 286 images, all 256x256 pixels.
- The **Waldos** are 5 cut-out pictures of Wally with a transparent background (PNG with an alpha channel).
- Our generated images will be saved in `/kaggle/working`, the folder Kaggle lets us write to.

In [ ]:
# Input data
background_directory = '/kaggle/input/datasets/zekikam/waldo-dataset/backgrounds/backgrounds'
waldo_directory = '/kaggle/input/datasets/zekikam/waldo-dataset/waldos/waldos'

# Where we will save the images we generate
output_directory = '/kaggle/working/generated images'
train_directory = os.path.join(output_directory, 'train images')
test_directory = os.path.join(output_directory, 'test images')

#Size of generated images (all input background images are 256x256)
image_size = 256

# How much data to generate
images_num = 5000 # in total
train_num = 4000 # 80% for training
test_num = 1000 # 20% for testing

# Model training settings
images_per_epoch = 1000 # the model will see the same 1000 training images every epoch
epochs_num = 10
model_batch_size = 32
learning_rate = 1e-3

### 0.2 Making our results repeatable 

We will be generating random backgrounds, random positions, random starting weights. 

Setting a **seed** means everyone running this notebook gets the same set of random numbers in the exact same order, so results can be compared (e.g. everyone's neural networks will start with the same exact initial weights).

We also pick the **device**: the GPU (`cuda`) if there is one, otherwise the CPU.

In [ ]:
seed = 1 
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    device = ... # TODO: Pick the PyTorch device for the GPU.
else:
    device = torch.device('cpu')

print('Using device:', device)

<hr>

# Step 1: Generating our own training data 🎨

To train a model to find Wally, we need lots of images for training, each with the known location of Wally. Labelling thousands of real pictures by hand would take days...

Instead, we will make our images, we:
- take a background
- paste Wally somewhere on it
- write down where we put him

This is called **synthetic data**!

<br>

### 1.1 Loading the backgrounds

Let's find all the background files and look at a few.

In [ ]:
# Takes each file in the background folder
# Combine the background directory with each file into a full path
# Puts all the paths in alphabetical order
background_paths = [] # holds all 286 paths
for this_file in os.listdir(background_directory):
    background_paths.append(os.path.join(background_directory, this_file))

background_paths = sorted(background_paths)
print('Number of backgrounds:', len(background_paths))


# Load them all into memory once, so we don't have to keep re-reading files from disk
# Resize to makes sure every background really is 256x256
backgrounds = [] # holds all 286 background images
for this_path in background_paths:
    image = Image.open(this_path).convert('RGB').resize((image_size, image_size))
    backgrounds.append(image)

print('Size of the first background:', backgrounds[0].size)

In [ ]:
# Let's have a look at 5 random backgrounds examples!
five_backgrounds = random.sample(backgrounds, 5)

plt.figure(figsize = (20,4))
for i in range(5):
    plt.subplot(1,5,i+1) #this figure has 1 row, 5 columns (5 images in total), and this plot is the (i+1)th plot
    plt.imshow(five_backgrounds[i])
    plt.axis('off')
    
plt.suptitle('5 random backgrounds')
plt.show()

**Think** 🤔

Why do we want lots of different backgrounds rather than one? What could go wrong if every training picture had the same background?

### 1.2 Loading the Waldos

Each Waldo is a PNG with 4 channels: **R**ed, **G**reen, **B**lue and **A**lpha. The alpha channel says how transparent each pixel is: 0 means fully transparent, 255 means its fully opaque. That's what lets us paste Wally without a white rectangle around him!

In [ ]:
# Same explanation as loading background images above
waldo_paths = []
for this_file in os.listdir(waldo_directory):
    waldo_paths.append(os.path.join(waldo_directory, this_file))

waldo_paths = sorted(waldo_paths)


# Convert('RGBA') makes sure every Waldo has an alpha channel
waldos = []
for this_file in waldo_paths:
    waldos.append(Image.open(this_file).convert(...)) # TODO: Which image mode keeps the alpha channel?

print('Dimensions of each waldo image')
for i in range(len(waldos)):
    print(os.path.basename(waldo_paths[i]), 'size (width, height):', waldos[i].size)

In [ ]:
# Show all the five Waldos
plt.figure(figsize = (3 * len(waldos), 4))
for i in range(len(waldos)):
    plt.subplot(1, len(waldos), i+1)
    plt.imshow(waldos[i])
    plt.title(f'Waldo {i}: {waldos[i].size[0]}x{waldos[i].size[1]}')
    plt.axis('off')
    
plt.suptitle('All 5 waldos')
plt.show()

### 1.3 Tweeking Wally a little bit each time 

If we pasted exactly the same 5 Waldos every time, the model might just memorise those exact pixels. In real images, each Wally is drawn differently each time, so to make our images as real as possible, we make every pasted Wally slightly different:

- a small rotation (up to ±10°)
- a brightness change (a bit darker or lighter)
- a colour shift (a little more or less red, green or blue)

This is called **data augmentation**.

We use NumPy for colour shifting: 
- turn the image into an array of numbers with shape `(height, width, 3)` - 3 because one number for each of R, B, G
- multiply the R, G and B channels by slightly different amounts, but leave the alpha channel alone

In [ ]:
def augment_waldo(wally):
    # 1. Rotate by a small random angle (between -10 and 10 degrees)
    # expand = True makes the canvas bigger so his hat and feet don't get cut off after rotation
    angle = ... # TODO: Pick a random rotation between -10 and 10 degrees.
    wally = wally.rotate(angle, expand=True)

    # Split Wally into R,G,B and Alpha (A - transparency)
    colours = wally.convert('RGB')
    alpha = wally.getchannel('A')

    # 2. Random brightness: 1.0 = unchanged, <1.0 = darker, >1.0 = brighter
    brightness = random.uniform(0.8, 1.2) # between these two chosen thresholds
    colours = ImageEnhance.Brightness(colours).enhance(brightness) # initialises and applies the birghtness enchancement object

    # 3. Random colour shift: make Wally slightly more or less red, green and blue
    pixels = np.array(colours).astype(float)
    # Format: pixels[row, col, colour]
    # : means - all of them
    pixels[:, :, 0] = pixels[:, :, 0] * random.uniform(0.9, 1.1) # red
    pixels[:, :, 1] = pixels[:, :, 1] * random.uniform(0.9, 1.1) # green
    pixels[:, :, 2] = pixels[:, :, 2] * random.uniform(0.9, 1.1) # blue
    pixels = np.clip(pixels, 0, 255).astype(np.uint8) # keep values between 0 and 255

    # Turn the numbers back into an image and add back the transparency
    wally = Image.fromarray(pixels)
    wally.putalpha(alpha)
    return wally

In [ ]:
# See what augmentation does to one Wally (6 times)
plt.figure(figsize = (15, 4))
for i in range(6):
    plt.subplot(1, 6, i + 1)
    plt.imshow(augment_waldo(waldos[0]))
    plt.axis('off')

plt.suptitle('One Wally, augmented 6 different ways!')
plt.show()

**Think** 🤔

- Why do we only rotate by a **small** amount? What would happen to the box if we rotated Wally by 90°?
- Can you think of an augmentation that would be a bad idea for finding Wally? (Hint: what if we turned his red stripes green?)

### 1.4 Pasting Wally onto a background 

Now we put it together. For one sample image we:

1. Pick a random background
2. Pick a random Wally and augment it
3. Pick a random top-left corner `(x, y)` so that Wally fits fully inside the picture
4. Paste him using his alpha channel as the **mask** (only the solid pixels are copied)
5. Work out his **bounding box**

<br>

**Bounding boxes** 📦

We describe a box with its top-left and bottom-right corners: `(x1, y1, x2, y2)`. This is the format `torchvision.ops.box_iou` wants.

In images, `x` counts pixels from the left and `y` counts pixels down from the top.

Because we rotated Wally, his picture now has transparent corners. `getbbox()` on the alpha channel gives the smallest box around the pixels that are actually Wally, so our box fits him tightly.

In [ ]:
def make_sample():
    # 1. Pick a random background and copy it, so we don't draw on the original because the 286 images are reused
    background = ... # TODO: Pick a random background and copy it so the original stays unchanged.

    # 2. Pick a random Wally and change him slightly 
    wally = augment_waldo(random.choice(waldos))
    wally_width, wally_height = wally.size

    # 3. Pick a random top-left corner where Wally fits completely inside the image
    x = random.randint(0, image_size - wally_width)
    y = random.randint(0, ...) # TODO: How far down can Wally go while still fitting inside the image?

    # 4. Paste Wally onto the background image
    # `mask=waldo` means only his non-transparent pixels are pasted
    background.paste(wally, (x, y), mask = ...) # TODO: Use the image whose alpha channel tells us which pixels to paste.

    # 5. Find the box around the visible part of Wally (i.e. where we pasted him)
    # wally.getchannel('A') takes the transparent layer
    # get_box() finds the smallest box around all the non-zero pixels (i.e. around Wally)
    # These four numbers measured from the top-left of Wally's own picture
    left, top, right, bottom = wally.getchannel('A').getbbox()
    # We want the top-left corner of Wally's picture to be at (x,y) of the background image
    box = ... # TODO: Turn the four local corners into (x1, y1, x2, y2) on the background.

    return background, box

**Think 🤔**

Why does `x` only go up to `image_size - wally_width` and not `image_size`?

### 1.5 Drawing boxes 

We will use **green** for the true box (where we put Wally) and later **red** for the box the model predicts.

Matplotlib's `patches.Rectangle` wants the top-left corner, then the width and height, so we convert from `(x1, y1, x2, y2)`.

In [ ]:
def draw_box(box, colour, ax = None):
    x1, y1, x2, y2 = box
    width = ... # TODO: Calculate the box width from its left and right corners.
    height = y2 -y1

    rectangle = patches.Rectangle((x1, y1), width, height, linewidth = 2, edgecolor = colour, facecolor = 'none')
    if ax is None:
        ax = plt.gca() # `gca` = get current axes
    ax.add_patch(rectangle)

In [ ]:
# Show 5 random generated images with a green box around Wally
plt.figure(figsize = (20, 4))
for i in range(5):
    image, box = make_sample()
    plt.subplot(1, 5, i + 1)
    plt.imshow(image)
    draw_box(box, 'green')
    plt.title(f'Box coordinates: {box}')
    plt.axis('off')

plt.show()

### 1.6 Making 5000 images and splitting them 80/20

We make all 5000 images up front and save them as files.

Then we split them:
- **Training set (80%, 4000 images):** the model will learn from these.
- **Test set (20%, 1000 images):** we keep these hidden until the end, to check the model works on pictures it has never seen.

`random_split` shuffles the numbers 0 to 4999 and deals them into two groups of 4000 and 1000. We give it its own seeded `generator` so the split is the same every time.

Each folder also gets a `labels.csv` file listing every image and the box around Wally in it. These are the correct answers, so during training we compare the model’s predicted box with them to calculate the loss, and during testing we use them to measure IoU.

`random_split(..., [a,b])` shuffles those numbers (...) and deals them into two piles so that no number lands in both piles: 
- the first `a` for training
- the next `b` for testing

In [ ]:
'''
Useful variables from Step 0:
images_num = 5000 # in total
train_num = 4000 # 80% for training
test_num = 1000 # 20% for testing
'''

import shutil
# Start with empty folders, so old images from a previous run don't get mixed in
shutil.rmtree(output_directory, ignore_errors=True)
os.makedirs(train_directory, exist_ok = True)
os.makedirs(test_directory, exist_ok = True)

# Decide which images go to training and which go to testing
train_ids, test_ids = random_split(range(images_num), [...] # TODO: Supply the two split sizes, in training then testing order.
    , generator = torch.Generator().manual_seed(seed))
test_ids = set(test_ids)
print('Training images:', len(train_ids))
print('Testing images:', len(test_ids))

In [ ]:
# Time how long it takes to generate all sample images
start_time = time.time()
# One row per training image: [file_name, x1, y1, x2, y2]
train_labels = [] 
test_labels = []

# Remember what images_num = ?
for i in range(images_num):
    image, box = make_sample() # go back to 1.4 if unsure
    x1, y1, x2, y2 = box
    file_name = f'image_{i:04d}.png' # e.g. image_0000.png

    if i in test_ids: # `test_ids` defined in cell above
        image.save(os.path.join(test_directory, file_name))
        test_labels.append([file_name, x1, y1, x2, y2])
    else: # i in train_id 
        image.save(os.path.join(train_directory, file_name))
        train_labels.append([file_name, x1, y1, x2, y2])

print(f'Made {images_num} images in {time.time()-start_time:1f} seconds.')

In [ ]:
# Saves the labels (the true boxes) as a CSV file in each folder
def save_labels(folder, labels):
    with open(os.path.join(folder, 'labels.csv'), 'w', newline='') as file:
        writer_object = csv.writer(file)
        writer_object.writerow(['File name', 'x1', 'y1', 'x2', 'y2']) # header row
        writer_object.writerows(labels)

# One `labels.csv` file each for training and testing data
save_labels(train_directory, train_labels)
save_labels(test_directory, test_labels)

# Subtract 1 because labels.csv is in the folder too
print(f'Train folder: {len(os.listdir(train_directory)) - 1} images')
print(f'Test folder: {len(os.listdir(test_directory)) - 1} images')

**Think** 🤔

Why must the test images never be used for training?

<hr>

# Step 2: Feeding images to PyTorch 🍼

A neural network only understands numbers, but PyTorch wants them in **tensors**. We need two helper classes from `torch.utils.data`:

- A **`Dataset`** answers two queries:
   - How many examples are there?: (`__len__`)
   - Give me example number `i`: (`__getitem__`)
    </br>
- A **`DataLoader`** takes a `Dataset` and serves it in shuffled batches:
  - it asks the Dataset for 32 random examples, stacks them into one tensor of shape (32, 3, 256, 256), and hands that to the training loop.
  - It also reshuffles every epoch, so the model never sees the images in the same order twice.
  - Training on a batch is much faster than one image at a time, and averaging the error over 32 images gives a steadier update than any single image would.


**What are tensors?**
</br>
Conceptually, a PyTorch tensor is identical to a NumPy ndarray, but tensors can run on GPUs and they can track gradients.

<br>

### 2.1 Our Wally `Dataset`

For example number `i`, our dataset:

1. Opens the image file
2. Turns it into a tensor of shape `(3, 256, 256)` (3 colour channels, 256 rows, 256 columns)
3. Scales pixel values from 0–255 down to 0–1 (smaller numbers are better for steady learning)
4. Scales the box from pixels down to 0–1 by dividing by the image size

<br>

**Why scale the box to 0–1?** 
</br>
Our model will output numbers that start out small (close to 0). If the targets were in pixels (up to 256), the model's first guesses would be off by about 100, the squared error would be about 10,000, and training would be wild and unstable. Keeping inputs and outputs on a small, similar scale makes learning smoother.

In [ ]:
# This class inherits from the torch.utils.data.Dataset class
# It tells PyTorch how to find and load our images
class WaldoDataset(Dataset):
    def __init__(self, folder):
        self.folder = folder
        self.rows = [] # will hold (file_name, box) for each image
        file = open(os.path.join(folder, 'labels.csv'))
        reader_object = csv.reader(file) # reads the csv file one row at a time
        next(reader_object) # skip the header row

        for row in reader_object:
            file_name = row[0] # e.g. image_0000.png
            # The csv stores numbers as text, so cast them into floats
            box = [float(row[1]), float(row[2]), float(row[3]), float(row[4])] # x1, y1, x2, y2
            self.rows.append((file_name, box))
            
        file.close()

    # Called when we write len(dataset)
    def __len__(self):
        return ... # TODO: Return the number of labelled images in the dataset.

    # Called when we write dataset[i]
    # It only loads an image from the disk when needed to save memory
    def __getitem__(self, i):
        file_name, box = self.rows[i]
        # 1. Open the image file, then make sure it has 3 colour channels
        image = Image.open(os.path.join(self.folder, file_name)).convert('RGB')
        # Convert PIL image to NumPy array of shape (256, 256, 3)
        image = np.array(image)
        # Convert NumPy array to PyTorch tensor of floats 
        image = torch.tensor(image, dtype=torch.float32)
        #PyTorch wants channels first, format: (3, 256, 256)
        image = ... # TODO: Move the colour channels first and scale the pixels from 0–255 to 0–1. # 3. divide by 255 so pixel values within the range(0,1)
        box = ... # TODO: Convert the box to a float32 tensor and scale its coordinates to 0–1. # 4.
        return image, box #its true box

In [ ]:
# One dataset for training, one for testing
# Same class but different folders
train_dataset = WaldoDataset(train_directory)
test_dataset = WaldoDataset(test_directory)
# calls __len__
print('Number of training images: ', len(train_dataset))
print('Number of test images: ', len(test_dataset))
print('\n')

# Check one example
image, box = train_dataset[0] # calls __getitem__(0)
print('One image tensor has shape:', image.shape) #expects (3, 256, 256)
print(f'Pixel values go from {image.min().item()} to {image.max().item()}') #expects 0 to 1
print('Its box (0-1):', box)

### 2.2 Using the same 1000 training images every epoch

An **epoch** is one pass through the training data. Our plan is 10 epochs of 1000 images, and every epoch uses the same 1000 images. We pick them once with `random_split` and keep the rest aside.

In [ ]:
leftovers = len(train_dataset) - images_per_epoch 

# This returns two values, but you only care about train_subset, ignore whatever's in _
# You can change images_per_epoch in Step 0, the higher it is, the more well-trained your model will be!
train_subset, _ = random_split(train_dataset, [images_per_epoch, leftovers], generator = torch.Generator().manual_seed(seed)) #generator = ... makes the shuffle the same every run, so you will always get the same 1000 images

print('Images used in each epoch:', len(train_subset))

### 2.3 Batches with a `DataLoader`

For each epoch, we don't feed images in one at a time, or all 1000 at once. We use **batches** of 32.

- One at a time is slow, and each update is based on just one noisy example.
- All at once might not fit in memory, and we'd only get one update per epoch.

**What are batches and epochs?**
- **Batch:** a small group of images (32 for us) that the model looks at together. It makes a guess for each one, we calculate the loss, and the weights get updated once per batch.
- **Epoch:** one full pass through all the training images. With 1000 images and batches of 32, one epoch is 32 batches, so 32 updates.

**Why batches of 32?**
Powers of 2 fit neatly into how GPU split up work, also smaller batches gives more weight updates per epoch, although each update is noiser because its averaged over fewer images. So 32 is a pretty good balance.

`shuffle=True` mixes up the order every epoch so the model doesn't learn anything from the order of the images. We don't shuffle the test set because the order doesn't matter there.

In [ ]:
# DataLoader grabs # = batch_size examples at a time and stack them into one tensor
train_loader = DataLoader(train_subset, batch_size = model_batch_size, shuffle = ... # TODO: Should training images be randomly reordered each epoch?
    )
test_loader = DataLoader(test_dataset, batch_size = model_batch_size, shuffle = False)

# Look at one batch
# Looping over a DataLoader gives one batch at a time
for images, boxes in train_loader:
    print(f'A batch of images has shape: {images.shape} -> (batch, channels, height, width)')
    print(f'A batch of boxes has shape: {boxes.shape} -> (batch, 4 numbers per box)')
    break

print('Batches per epoch:', len(train_loader))

**Think** 🤔

The weights are updated once per batch. How many updates happen in 10 epochs?

<hr>

# Step 3: Building our CNN from scratch 🧠

Our model has two parts:

1. **The feature extractor (convolution + pooling):** turns the picture into a small grid of "features" (things like *red and white stripes here*, *round glasses there*).
2. **The head:** turns those features into 4 numbers, the corners of Wally's box `(x1, y1, x2, y2)`.

<br>

### Part 1 The building block: Conv → ReLU → MaxPool

- **`nn.Conv2d(in, out, 3, padding=1)`** slides `out` small 3x3 filters (kernels) over the image. Each filter learns to spot one pattern. The **same** filter is used at every position, so a stripe detector works wherever Wally is. `padding=1` keeps the width and height the same.
- **`nn.ReLU()`** keeps positive values and sets negatives to 0. Without it, stacking layers would be no more powerful than a single layer.
- **`nn.MaxPool2d(2)`** keeps the largest value in each 2x2 square. This halves the width and height, so later layers see a bigger area of the picture and there's less to compute.

<br>

### How the size changes

| After | Shape (channels x height x width) |
|---|---|
| | 3 x 256 x 256 |
| block 1 (16 filters) | 16 x 128 x 128 |
| block 2 (32 filters) | 32 x 64 x 64 |
| block 3 (64 filters) | 64 x 32 x 32 |
| block 4 (64 filters) | 64 x 16 x 16 |

Each MaxPool halves the size, and there are 4 of them, so the final grid is `image_size // 16` = 16 cells across. We store that in `feature_size` so the model still works if the image size changes.

Think of the 16x16 grid as cutting the picture into 256 squares, each 16x16 pixels. For every square we have 64 numbers describing what's in it.

### Part 2 The head: "Where's Wally?" as a heatmap

**A. heatmap(x) function**
- **Input:** A batch of images (B, 3, 256, 256), where B = batch size (32)
- **Run the conv blocks:** this gives (B, 64, 16, 16), a 16x16 grid with 64 pattern numbers per patch
- **Score each patch in the grid:** the 1x1 conv squashes each patch's 64 numbers into 1 score for "how Wally-like" is this?, giving (B, 1, 16, 16)
- **Flatten scores:** lay the 256 scores out in one list per image (B, 256), so softmac can threat them as one group
- **Softmax:** turn the scores into probabilities (0 to 1, adding up to 1 per image). So the patch with the highest score will have the highest probability
- **Reshape:** turn the list of probabilities back into a 16x16 grid, which will be the heatmap
- **Output:** the heatmap and features because `forward` needs them for the size ("features" are the 4 conv blocks output from Part 1, the 16x16 grid for each image that describes the image)


**B. forward(x) function**
So far, the heatmap only answers where Wally is (the center), but to bound Wally with a box, we also need to know how big he is. So `forward` goes back to the features to work out the size.

**For each row and column of the heatmap:**

- **Column chances:** add down each column to get the chance Wally is in that column
- **Row chances:** add across each row to get the chance Wally is in that row
- **Center x and y:** take the weighted average of the square positions, weighted by its probability
- **Strongest features:** take the largest value across the grid for each of the 64 channels
- **Size:** `Linear(64, 2)` turns those 64 numbers into width and height, and sigmoid keeps them between 0 and 1
- **Corners:** center ± half the size gives the box corners (x1, y1, x2, y2), then clamp them to 0–1
- **Output:** (B, 4) exactly 4 numbers per image, (x1, y1, x2, y2), between 0 and 1. So the loss, training loop and IoU are unchanged.

<br>

**Shapes inside `forward`** (with a batch of `B` images):

| Variable | Shape | Meaning |
|---|---|---|
| `features` | B x 64 x 16 x 16 | 64 numbers for each of the 256 squares |
| `scores` | B x 256 | one score per square |
| `heatmap` | B x 16 x 16 | probabilities, adding up to 1 |
| `heatmap.sum(dim=1)` | B x 16 | probability of each **column**, which gives `x` |
| `heatmap.sum(dim=2)` | B x 16 | probability of each **row**, which gives `y` |
| `width`, `height` | B | box size, as a fraction of the image |
| output | B x 4 | `(x1, y1, x2, y2)` |

In [ ]:
# There are 4 MaxPool layers, each halving the size: 256 -> 128 -> 64 -> 32 -> 16
# Halving 4 times = dividing by 16, so the final grid is 16x16 squares
# where each square covers a 16x16 pixel patch of the original image!
feature_size = image_size // 16

# WaldoFinder inherits the class torch.nn.Module, which is the base class for all neural network modules
# Runs once per image
class WaldoFinder(nn.Module):
    def __init__(self):
        super().__init__() # sets up nn.Module so PyTorch can track your layers
        
        '''
        Part 1: the feature extractor
        4 blocks of Conv -> ReLU -> MaxPool
        '''
        self.features = nn.Sequential( # a container module that lets you stack layers in a linear order so that the output of each layer automatically becomes the input for the next layer
            # nn.Conv2d(input_image_channels_num, output_image_channels_num, kernel size, padding (default=0 zero padded on all four sides of the input image))
            # nn.Conv2d(3, 16, ...) means: take the 3 colour channels in and run 16 different filters over them
            # The model will learn what the 16 patterns are (typically the first layer will detect a horizontal or vertical edge, red here, etc.)
            # First block: 3 colours in -> 16 simple patterns (edges, colours)
            nn.Conv2d(..., 16, kernel_size = 3, padding = 1), nn.ReLU(), nn.MaxPool2d(2), # TODO: How many colour channels does the input image have? # 256x256 -> 128x128
            # Second block: 16 -> 32 patterns built from block 1's (e.g. stripes, corners)
            nn.Conv2d(16, 32, kernel_size = 3, padding = 1), nn.ReLU(),nn.MaxPool2d(2), # 128x128 -> 64x64
            # Third : 32 -> 64 more complex patterns (e.g. striped shirt, hat shape)
            nn.Conv2d(32, 64, kernel_size = 3, padding = 1), nn.ReLU(), nn.MaxPool2d(2), # 64x64 -> 32x32
            # Fourth block: 64 -> 64 patterns covering bigger areas (e.g. whole-Wally parts)
            nn.Conv2d(64, 64, kernel_size = 3, padding = 1), nn.ReLU(), nn.MaxPool2d(2) # 32x32 -> 16x16
        )
         # Result: a 16 x 16 grid = 256 patches, where every patch has 64 numbers syaing which patterns were found in it
         # This is not a picture! It's a map of description
         # The grid shrinks as the filters grow because each MaxPool zooms out, so the next convolution sees a wider area of the original


        '''
        Part 2: creating the "head" that turns features into a box
        '''
        # Head part 1: one "how Wally-like?" score for every patch of the grid 
        # A 1x1 convolution looks at each patch of the grid and squashes it into a score
        # It needs to learn what Wally looks like once
        self.score = nn.Conv2d(64, ..., kernel_size = 1) # TODO: We want one Wally score per grid square. How many output channels?

        # Head part 2: Wally's width and height
        # Takes 64 numbers (one per pattern detector) and outputs 2 numbers
        self.size = nn.Linear(64, ...) # TODO: How many numbers describe width and height?

        # The centre position of each grid square, as a fraction of the image (0 to 1)
        square_centres = (torch.arange(feature_size) + 0.5) / feature_size
        # register_buffer means it moves to the GPU with the model, but is not a weight to train
        self.register_buffer('square_centres', square_centres)

    #This heatmap is a 16x16 grid of probabilities (each representing where's Wally?)
    def heatmap(self, x):
        features = ... # TODO: Pass the images through the feature extractor. Output: (B, 64, 16, 16).
        scores = self.score(features).flatten(start_dim = 1) # (B, 256)
        probs = ... # TODO: Turn each image's grid scores into probabilities that add up to 1.  # (B, 256), adds up to 1
        heatmap = probs.view(-1, feature_size, feature_size)
        return heatmap, features

    def forward(self, x):
        heatmap, features = self.heatmap(x)

        #dim 1 = rows, dim 2 = columns
        # sum(dim = 1) adds down each column to calculate the chance Wally is in each column
        column_probs = heatmap.sum(dim = 1) # (B, 16) because There are 16 columns in a 16x16 grid
        row_probs = ... # TODO: Add across the columns to get one probability per row. # (B, 16)

        center_x = ... # TODO: Calculate the weighted average of the column centres using column_probs.
        center_y = (row_probs * self.square_centres).sum(dim=1)

        # Work out how big is this Wally
        # Take the strongest feature out of the 64 features/ patterns
        strongest = features.amax(dim = (2, 3))
        size = self.size(strongest).sigmoid() # keep width and height positive and between 0 and 1 
        width = size[:, 0] # first number = width
        height = size[:, 1] # second number = height

        # centre + size = box corners 
        # The box goes half a width left and right of the center and half a height above and below it
        x1 = ... # TODO: Calculate the left corner from the centre and width.
        y1 = center_y - height / 2 #top
        x2 = center_x + width / 2 #right
        y2 = ... # TODO: Calculate the bottom corner from the centre and height.
        
        # (B, 4), same format as our labels
        return torch.stack([x1, y1, x2, y2], dim=1).clamp(0, 1) # keep corners inside the image

In [ ]:
# WaldoFinder() builds all the layers with random starting weights
model = WaldoFinder().to(device) # Use GPU
print(model) #Lists every layer in order

num_params = 0
for p in model.parameters():
    num_params += p.numel()

print(f'\nNumber of weights and biases: {num_params:,}')

### Checking the shapes 

Before training, it's good practice to push a fake batch through the model and check the shapes are what we expect: a 16x16 heatmap and one box of 4 numbers for each image.

In [ ]:
fake_batch = torch.zeros(2, 3, image_size, image_size).to(device)
heatmap, features = model.heatmap(fake_batch)
print('Feature grid shape:', features.shape) # expect (2, 64, 16, 16)
print('Heatmap shape:', heatmap.shape, 'each heatmap adds up to', heatmap[0].sum().item()) # expect (2, 16, 16) and 1.0
print('Output shape:', model(fake_batch).shape) # expect (2, 4): one box per image

<hr>

# Step 4: The loss function and the optimiser 📉

### Loss: how wrong is the model?

The loss turns "how bad is this prediction?" into a single number. Training means making this loss as small as possible.

We use **mean squared error (MSE)**: for every image, take the 4 predicted numbers, subtract the 4 true numbers, square the differences and average them all.

$$\text{MSE} = \frac{1}{N}\sum (\text{predicted} - \text{true})^2$$

Squaring makes every error positive and punishes big mistakes much more than small ones.

<br>

### Optimiser: how do we fix it?

The optimiser decides how to nudge each weight to reduce the loss. We use **Adam (Adaptive Moment Estimation)**, which adapts its step size for each weight separately and usually works well without much tuning.

The learning rate is how big those nudges are. If it's too big, there'll be a high risk of overshoot, but if it's too small, it'll takes forever. We use 0.001.

In [ ]:
# How wrong is the predicted box?
# MSE = average of (predicted - true) squared, so 0 means perfect
loss_function = ... # TODO: Create the mean squared error loss function.

# Nudges all weights to make the loss smaller
# lr = how big each nudge is
# Adam adjusts the step size for each weight automatically
optimizer = torch.optim.Adam(..., lr = learning_rate # TODO: Give Adam the model parameters it should update.
    )

**Think** 🤔

- Our boxes are scaled 0–1. If the model is off by 0.1 on every coordinate, what is the MSE? How many pixels is that?
- Why might a learning rate that's too large make training get "stuck"?

<hr>

# Step 5: Training the model 🏋️‍♂️

**For every batch:**

1. **Forward pass:** `predictions = model(images)`. The images go through the network and outputs predicted boxes
2. **Loss:** `loss = loss_function(predictions, boxes)`
3. **`optimizer.zero_grad()`:** clear the gradients from the previous batch. PyTorch adds new gradients onto old ones, so if we forget this, old batches keep affecting new updates
4. **`loss.backward()`:** backpropagation. PyTorch works out, for every weight, which direction would make the loss smaller (the gradient).
5. **`optimizer.step()`:** nudge every weight a little in that direction.

Since the weights are updated once per batch and there are 32 batches and 10 epochs: 32 times per epoch, 320 times in total.

<br>

After each epoch we also check the loss on the test set, without updating any weights. If the training loss keeps falling but the test loss starts rising, the model is memorising the training images rather than learning to find Wally (**overfitting**).

- `model.train()` and `model.eval()` switch the model between training and testing mode
- `torch.no_grad()` tells PyTorch not to track gradients, which saves memory and time when we're only testing

In [ ]:
def test_loss():
    model.eval() #switch to testing mode
    total = 0
    with ...: # TODO: Disable gradient tracking while evaluating the model. 
        for images, boxes in test_loader:
            images = images.to(device)
            boxes = boxes.to(device)
            predictions = model(images) # model = WaldoFinder()
            loss = loss_function(predictions, boxes)

            # This loss is the batch's average, so multiply by the batch size to get the batch totla loss
            total += loss.item() * len(images)
    return total / len(test_dataset)

In [ ]:
train_losses = []
test_losses = []
start_time = time.time()

for epoch in range(epochs_num):
    model.train() #switch to training mode
    running_loss = 0

    for images, boxes in train_loader:
        images = images.to(device)
        boxes = boxes.to(device)

        predictions = ... # TODO: 1. Pass the batch of images through the model.
        loss = ... # TODO: 2. Compare the predicted boxes with the true boxes using our loss function.

        ... # TODO: 3. Clear the gradients from the previous batch.
        ... # TODO: 4. Run backpropagation to calculate the gradients.
        ... # TODO: 5. Ask the optimiser to update the model weights.

        running_loss += loss.item() * len(images)

    # Average loss over the whole epoch
    train_losses.append(running_loss / len(train_subset))
    test_losses.append(test_loss())

    seconds = time.time() - start_time
    print('Epoch', epoch + 1, '/', epochs_num,
          ' train loss:', round(train_losses[-1], 4),
          ' test loss:', round(test_losses[-1], 4),
          ' time:', round(seconds), 's')

minutes = (time.time() - start_time) / 60
print('Training took', round(minutes, 1), 'minutes')

### Plotting the learning curves 📈

A **learning curve** shows the loss after each epoch. We want both lines to go down. A big gap where the test loss is much higher than the training loss is a sign of overfitting.

In [ ]:
# Plot how the loss changed over the epochs
epochs = range(1, epochs_num + 1)
plt.figure(figsize = (8, 5))
plt.plot(epochs, train_losses, 'o-', label = 'training loss')
plt.plot(epochs, test_losses, 'o-', label = 'test loss')
plt.xlabel('epoch')
plt.ylabel('mean squared error')
plt.title('Learning curves')
plt.legend()
plt.show()

**Think** 🤔

- What would happen if we deleted `optimizer.zero_grad()`? Try it!
- Is there a gap between the training and test loss? What does that tell you?

<hr>

# Step 6: How good are the boxes? IoU 📏

The loss tells us the model is improving, but "MSE = 0.0002" is hard to picture. 

A better question is: how much do the predicted and true boxes overlap?

<br>

### Intersection over Union (IoU)

$$\text{IoU} = \frac{\text{Area where the two boxes overlap}}{\text{Total area covered by the two boxes together}}$$

- 1 means a perfect match
- 0 means the boxes don't touch at all
- A common rule of thumb: IoU > 0.5 counts as a correct detection

<br>

`torchvision.ops.box_iou(boxes1, boxes2)` takes boxes as `(x1, y1, x2, y2)` and returns a table comparing **every** box in the first list with **every** box in the second. We only want each prediction compared with its **own** true box, so we take the **diagonal** of that table.

In [ ]:
# A quick example with two boxes that we can work out by hand
true_box = torch.tensor([[0., 0., 10., 10.]])   # a 10x10 square
pred_box = torch.tensor([[5., 0., 15., 10.]])   # the same square moved 5 pixels right
print('IoU =', box_iou(true_box, pred_box).item())

In [ ]:
model.eval()
all_predictions, all_boxes = [], []
with torch.no_grad():
    for images, boxes in test_loader:
        all_predictions.append(model(images.to(device)).cpu())
        all_boxes.append(boxes)

# back to pixels
pred_boxes = torch.cat(all_predictions) * image_size
true_boxes = torch.cat(all_boxes) * image_size

ious = ... # TODO: Calculate box IoU and take the diagonal so each prediction is compared with its own true box.
print(f'Mean IoU on the {len(ious)} test images: {ious.mean():.2f}')
print(f'Correct detections (IoU > 0.5): {(ious > 0.5).float().mean():.1%}')

In [ ]:
plt.figure(figsize = (8, 5))
plt.hist(ious.numpy(), bins = 20, range = (0, 1))
plt.axvline(0.5, color = 'red', linestyle = '--', label = 'IoU = 0.5')
plt.xlabel('IoU')
plt.ylabel('number of test images')
plt.title('How well do the predicted boxes overlap the true boxes?')
plt.legend()
plt.show()

<hr>

# Step 7: Testing on unseen images 👀

The moment of truth! These are test images the model has never trained on.

- **Green box:** where Wally really is
- **Red box:** where the model thinks he is

Under each picture we show the IoU and the error: how many pixels the centre of the red box is from the centre of the green box.

Run the cell a few times to see different images.

In [ ]:
def box_centre(box):
    x1, y1, x2, y2 = box
    return ... # TODO: Return the box centre as (x, y) by averaging each pair of corners.

fig, axes = plt.subplots(2, 4, figsize = (20, 10))
for ax, i in zip(axes.flatten(), random.sample(range(len(test_dataset)), 8)):
    image, _ = test_dataset[i]
    true_box, pred_box = true_boxes[i].tolist(), pred_boxes[i].tolist()

    (tx, ty), (px, py) = box_centre(true_box), box_centre(pred_box)
    error = ((tx - px) ** 2 + (ty - py) ** 2) ** 0.5

    ax.imshow(image.permute(1, 2, 0))       # back to (height, width, channels) for plotting
    draw_box(true_box, 'green', ax)
    draw_box(pred_box, 'red', ax)
    ax.set_title(f'IoU = {ious[i]:.2f}   error = {error:.1f} px')
    ax.axis('off')
plt.show()

### Where is the model looking? 🔥

We can also draw the **heatmap**, the model's probability for each of the 256 squares, on top of the picture. Bright squares are where the model thinks Wally is.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize = (20, 5))
for ax, i in zip(axes, random.sample(range(len(test_dataset)), 4)):
    image, _ = test_dataset[i]
    with torch.no_grad():
        heatmap, _ = model.heatmap(image.unsqueeze(0).to(device))   # unsqueeze: a batch of 1

    ax.imshow(image.permute(1, 2, 0))
    # stretch the 16x16 heatmap over the 256x256 picture
    # alpha (see-through-ness) follows the probability, so unlikely squares are invisible
    heat = heatmap[0].cpu()
    ax.imshow(heat, cmap = 'autumn', alpha = (heat / heat.max()).numpy(), extent = (0, image_size, image_size, 0))
    draw_box(true_boxes[i].tolist(), 'green', ax)
    ax.set_title(f'most likely square: {heatmap[0].max():.0%} sure')
    ax.axis('off')
plt.show()

### The hardest test images 😬

It's often more useful to look at where the model **fails**. These are the 4 test images with the lowest IoU.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize = (20, 5))
for ax, i in zip(axes, ious.argsort()[:4].tolist()):
    image, _ = test_dataset[i]
    ax.imshow(image.permute(1, 2, 0))
    draw_box(true_boxes[i].tolist(), 'green', ax)
    draw_box(pred_boxes[i].tolist(), 'red', ax)
    ax.set_title(f'IoU = {ious[i]:.2f}')
    ax.axis('off')
plt.show()

**Think** 🤔

- *Look at the worst images. Why do you think the model got these wrong? Is it the background, the Waldo, or where he was placed?*
- *Our model always outputs exactly one box. What would it do with a picture that has **no** Wally in it? With **two** Wallys?*
- *We trained only on pasted Wallys. Do you expect it to work on a real page from a Where's Wally book? Why or why not?*

<hr>

# What's next? 🚀

Some things to try if you somehow manage to finish early:

- Train on all 4000 images (`images_per_epoch = 4000`) or for more epochs. Does the IoU go up?
- Change the learning rate to `1e-2` or `1e-5`. What happens to the learning curves?
- Replace the heatmap head with `nn.Flatten()` → `nn.Linear(64 * feature_size * feature_size, 64)` → `nn.ReLU()` → `nn.Linear(64, 4)`. Does it still learn?
- Add stronger augmentation, like bigger rotations or making Wally smaller. Does the model still find him?
- Add a second head that answers "is Wally in this picture at all?" and train it on some images without Wally.